# Fix the dicom files

In [1]:
from tqdm import tqdm
from pathlib import Path

import numpy as np
import pandas as pd

import nibabel as nib
from nibabel.processing import resample_from_to
import pydicom

import json

In [2]:
root_dir = Path('/data/vision/polina/projects/fetal/common-data/BRAIN-IQA/dataset1/')
save_loc = Path('/data/vision/polina/users/marcusbl/data')

label_map = {
            '{"image_quality":"bad"}': 0,
            '{"image_quality":"good"}': 1
} 

In [ ]:
for person_path in list(root_dir.iterdir()):

    if not person_path.is_dir():
        continue

    for stack_path in person_path.iterdir():
        if not stack_path.is_dir(): # not a stack folder
            continue

        # 1) Parse CSV File for Labels
        csv_file = next(stack_path.glob("*.csv"), None)
        if csv_file is None:
            continue

        label_df = pd.read_csv(csv_file)

        # 2) Align Nifti & Dicom Orders
        dicoms_dir_path = stack_path / "dicoms"
        nifti_scan_path = stack_path / 'converted.nii.gz'
        nifti_mask_path = stack_path / 'converted_mask.nii.gz'

        nifti_scan = nib.load(nifti_scan_path)
        nifti_mask = nib.load(nifti_mask_path)
        
        # 3) Fix Ordering of Dicom Files
        sorted_dicom_files = sorted(dicoms_dir_path.glob("*.dcm"), key = lambda s: s.name[:4], reverse=True)
        first_dicom = pydicom.dcmread(sorted_dicom_files[0]).pixel_array.astype(dtype=np.float32)
        first_nifti =  np.rot90(np.asarray(nifti_scan.dataobj[:, :, 0]), k = 1, axes = (0,1))

        if not np.allclose(first_dicom, first_nifti): # don't match -> reverse order
            sorted_dicom_files.reverse()

        # 4) Create Aligned Dicom & Nifti w/ Masks  
        dicom_stack = np.stack([pydicom.dcmread(dcm_file).pixel_array.astype(dtype=np.float32) for dcm_file in sorted_dicom_files], axis = -1)
        nifti_stack = nifti_scan.get_fdata(dtype=np.float32)
        mask_stack = resample_from_to(nifti_mask, nifti_scan, order = 0).get_fdata(dtype=np.float32)
        
        # rot each by 90 degrees
        nifti_stack = np.rot90(nifti_stack, k=1, axes=(0,1))
        mask_stack = np.rot90(mask_stack, k=1, axes=(0,1))

        assert np.allclose(dicom_stack, nifti_stack)
        
        # 5) Create Labeled JSON
        dicom_labels = {}
        old_mapping = {}
        for scan_num, fpath in enumerate(sorted_dicom_files):
            # Get the label
            row = label_df.loc[label_df["External ID"] == (fpath.stem + ".png")]
            label_str = row["Label"].values[0]

            # If unknown label -> skip
            if label_str not in label_map: 
                continue
            
            dicom_labels[scan_num] = label_map[label_str]
            old_mapping[scan_num] = str(fpath)

        if len(dicom_labels) == 0:
            continue

        # 6) Save Outputs
        clean_folder = save_loc / '/'.join(stack_path.parts[-2:]) / 'clean'
        clean_folder.mkdir(exist_ok=True, parents=True)

        with open(clean_folder / 'labels.json', 'w') as f:
            json.dump(dicom_labels, f)
        with open(clean_folder / 'old_maps.json', 'w') as f:
            json.dump(old_mapping, f)

        np.save(clean_folder / 'dicoms.npy', dicom_stack)
        np.save(clean_folder / 'niftis.npy', nifti_stack)
        np.save(clean_folder / 'masks.npy', mask_stack)
        
        print(f"Saved to {clean_folder}")


(256, 256, 35)
(256, 256, 43)
(256, 256, 39)
(256, 256, 20)


KeyboardInterrupt: 